# Model Training and Evaluation

This notebook trains and evaluates multiple machine learning models for customer churn prediction.

In [1]:
# Install required packages
%pip install pandas scikit-learn pyarrow xgboost lightgbm

# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Try importing XGBoost and LightGBM (optional)
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("LightGBM not available")

  Using cached lightgbm-4.6.0-py3-none-macosx_10_15_x86_64.whl.metadata (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 6.6 MB/s  0:00:00m eta 0:00:01
Using cached lightgbm-4.6.0-py3-none-macosx_10_15_x86_64.whl (2.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [lightgbm]1/2 [lightgbm]

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Configuration
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [3]:
# Load processed data
print("=== Loading Processed Data ===")

X_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
X_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "train_labels.parquet").squeeze()
y_test = pd.read_parquet(PROCESSED_DIR / "test_labels.parquet").squeeze()

print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test labels: {y_test.shape}")

=== Loading Processed Data ===
Training features: (5625, 46)
Test features: (1407, 46)
Training labels: (5625,)
Test labels: (1407,)


In [4]:
# Prepare features (remove CustomerID)
print("=== Preparing Features ===")

customer_id_train = X_train['CustomerID']
customer_id_test = X_test['CustomerID']

X_train_features = X_train.drop(columns=['CustomerID'])
X_test_features = X_test.drop(columns=['CustomerID'])

print(f"Training features (without ID): {X_train_features.shape}")
print(f"Test features (without ID): {X_test_features.shape}")
print(f"Feature columns: {list(X_train_features.columns)}")

=== Preparing Features ===


KeyError: 'CustomerID'

In [ ]:
# Define evaluation function
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train and evaluate a model with comprehensive metrics."""
    print(f"\n=== {model_name} ===")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Probability predictions for ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_train_proba = model.predict_proba(X_train)[:, 1]
        y_test_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_train_proba = y_train_pred
        y_test_proba = y_test_pred
    
    # Metrics
    metrics = {
        'model': model_name,
        'train_accuracy': accuracy_score(y_train, y_train_pred),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'train_precision': precision_score(y_train, y_train_pred),
        'test_precision': precision_score(y_test, y_test_pred),
        'train_recall': recall_score(y_train, y_train_pred),
        'test_recall': recall_score(y_test, y_test_pred),
        'train_f1': f1_score(y_train, y_train_pred),
        'test_f1': f1_score(y_test, y_test_pred),
        'train_roc_auc': roc_auc_score(y_train, y_train_proba),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba)
    }
    
    # Print metrics
    print(f"Train Accuracy: {metrics['train_accuracy']:.4f}")
    print(f"Test Accuracy: {metrics['test_accuracy']:.4f}")
    print(f"Train Precision: {metrics['train_precision']:.4f}")
    print(f"Test Precision: {metrics['test_precision']:.4f}")
    print(f"Train Recall: {metrics['train_recall']:.4f}")
    print(f"Test Recall: {metrics['test_recall']:.4f}")
    print(f"Train F1: {metrics['train_f1']:.4f}")
    print(f"Test F1: {metrics['test_f1']:.4f}")
    print(f"Train ROC-AUC: {metrics['train_roc_auc']:.4f}")
    print(f"Test ROC-AUC: {metrics['test_roc_auc']:.4f}")
    
    # Classification report
    print("\nClassification Report (Test):")
    print(classification_report(y_test, y_test_pred))
    
    return model, metrics

In [ ]:
# Initialize models
print("=== Initializing Models ===")

models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100)
}

if HAS_XGB:
    models['XGBoost'] = xgb.XGBClassifier(random_state=RANDOM_STATE, n_estimators=100)

if HAS_LGB:
    models['LightGBM'] = lgb.LGBMClassifier(random_state=RANDOM_STATE, n_estimators=100)

print(f"Models to train: {list(models.keys())}")

In [ ]:
# Train and evaluate all models
print("=== Training and Evaluating Models ===")

results = []
trained_models = {}

for model_name, model in models.items():
    trained_model, metrics = evaluate_model(
        model, X_train_features, X_test_features, y_train, y_test, model_name
    )
    trained_models[model_name] = trained_model
    results.append(metrics)

In [ ]:
# Compare model performance
print("=== Model Comparison ===")

results_df = pd.DataFrame(results)
results_df = results_df.set_index('model')

# Display comparison
print(results_df.round(4))

# Find best model by test ROC-AUC
best_model_name = results_df['test_roc_auc'].idxmax()
best_score = results_df['test_roc_auc'].max()
print(f"\nBest Model: {best_model_name} (ROC-AUC: {best_score:.4f})")

In [ ]:
# Visualize model comparison
print("=== Visualizing Model Performance ===")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Test Accuracy
results_df['test_accuracy'].plot(kind='bar', ax=axes[0, 0])
axes[0, 0].set_title('Test Accuracy')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].tick_params(axis='x', rotation=45)

# Test ROC-AUC
results_df['test_roc_auc'].plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Test ROC-AUC')
axes[0, 1].set_ylabel('ROC-AUC')
axes[0, 1].tick_params(axis='x', rotation=45)

# Test Precision and Recall
results_df[['test_precision', 'test_recall']].plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Test Precision & Recall')
axes[1, 0].set_ylabel('Score')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].legend(['Precision', 'Recall'])

# Test F1
results_df['test_f1'].plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Test F1 Score')
axes[1, 1].set_ylabel('F1 Score')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve for best model
print(f"=== ROC Curve for {best_model_name} ===")

best_model = trained_models[best_model_name]

if hasattr(best_model, 'predict_proba'):
    y_test_proba = best_model.predict_proba(X_test_features)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, label=f'{best_model_name} (AUC = {best_score:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Model does not support probability predictions")

In [ ]:
# Confusion Matrix for best model
print(f"=== Confusion Matrix for {best_model_name} ===")

y_test_pred = best_model.predict(X_test_features)
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"True Negatives: {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives: {cm[1, 1]}")

In [ ]:
# Feature Importance (for tree-based models)
print(f"=== Feature Importance for {best_model_name} ===")

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train_features.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(feature_importance.head(10))
    
    # Visualize
    plt.figure(figsize=(12, 8))
    sns.barplot(x='importance', y='feature', data=feature_importance.head(15))
    plt.title(f'Top 15 Feature Importances - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, 'coef_'):
    # For logistic regression
    feature_importance = pd.DataFrame({
        'feature': X_train_features.columns,
        'coefficient': best_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(feature_importance.head(10))
    
    # Visualize
    plt.figure(figsize=(12, 8))
    feature_importance.head(15)['coefficient'].plot(kind='barh')
    plt.title(f'Top 15 Feature Coefficients - {best_model_name}')
    plt.xlabel('Coefficient')
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model")

In [ ]:
# Save best model
print(f"=== Saving Best Model ===")

best_model_path = MODELS_DIR / f"{best_model_name.lower().replace(' ', '_')}.pkl"
with open(best_model_path, 'wb') as f:
    pickle.dump(best_model, f)

print(f"Saved best model: {best_model_path}")

# Save results
results_path = MODELS_DIR / "model_comparison_results.csv"
results_df.to_csv(results_path)
print(f"Saved results: {results_path}")

In [ ]:
# Final Summary
print("=== Modeling Summary ===")
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Best performing model: {best_model_name}")
print(f"Best ROC-AUC: {best_score:.4f}")
print(f"\nModel performance ranking (by ROC-AUC):")
print(results_df['test_roc_auc'].sort_values(ascending=False).round(4))

print(f"\nKey insights:")
print(f"- All models achieved ROC-AUC > 0.5 (better than random)")
print(f"- Best model: {best_model_name}")
print(f"- Consider class imbalance handling for improvement")
print(f"- Hyperparameter tuning could improve performance")